In [1]:
!python -m pip install --user datasets


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from datasets import load_dataset


C:\Users\mansh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# !pip install -U g4f
# !pip install aiohttp
!python -m pip install --user -U g4f
!python -m pip install --user aiohttp

  Obtaining dependency information for g4f from https://files.pythonhosted.org/packages/7b/63/73d91b25b2304967ce1249f7eb911ae128a8c6fa7dccf5db8d141adf8dcf/g4f-0.4.3.5-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/54.4 kB ? eta -:--:--
     ---------------------------------------- 54.4/54.4 kB 1.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ------------ --------------------------- 0.3/1.2 MB 7.2 MB/s eta 0:00:01
   --------------------------------- ------ 1.0/1.2 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 9.1 MB/s eta 0:00:00
  Attempting uninstall: g4f
    Found existing installation: g4f 0.4.3.1
    Uninstalling g4f-0.4.3.1:
      Successfully uninstalled g4f-0.4.3.1



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import time
from g4f.client import Client

# Initialize the client
client = Client()

# List of input messages
messages = [
    "Tell me a joke on ML engineer"
]

total_time = 0
retry_limit = 3  # Maximum retries for a failed request

# Loop through each message, send it, and measure response time
for i, msg in enumerate(messages):
    print(f"Processing message {i + 1} of {len(messages)}...")

    for attempt in range(retry_limit):
        try:
            start_time = time.time()  # Start the timer

            # API request
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": msg}],
                web_search=False
            )

            end_time = time.time()  # End the timer
            elapsed_time = end_time - start_time
            total_time += elapsed_time

            print(f"Message {i + 1}: {msg}")
            print(f"Response: {response.choices[0].message.content}")
            print(f"Time taken: {elapsed_time:.2f} seconds\n")
            break  # Exit retry loop on success

        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt + 1 == retry_limit:
                print(f"Skipping message {i + 1} after {retry_limit} attempts.\n")
            else:
                print("Retrying...\n")
                time.sleep(1)  # Add delay before retrying

# Summary
print(f"Total time for {len(messages)} messages: {total_time:.2f} seconds")
if messages:
    print(f"Average time per message: {total_time / len(messages):.2f} seconds")


Processing message 1 of 1...
Attempt 1 failed with error: RetryProvider failed:
PollinationsAI: RuntimeError: asyncio.run() cannot be called from a running event loop
Blackbox: RuntimeError: asyncio.run() cannot be called from a running event loop
Liaobots: RuntimeError: asyncio.run() cannot be called from a running event loop
ChatGptEs: RuntimeError: asyncio.run() cannot be called from a running event loop
Copilot: MissingRequirementsError: Install or update "curl_cffi" package | pip install -U curl_cffi
OpenaiChat: NoValidHarFileError: har_and_cookies dir is not readable
ChatGptt: ClientConnectorCertificateError: Cannot connect to host chatgptt.me:443 ssl:True [SSLCertVerificationError: (1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1002)')]
Jmuz: ResponseError: Service currently unavailable in your region. Sorry!
DarkAI: ResponseStatusError: Response 429: event: error
data: {"event": "error", "data": {"event_type": "er

c:\Users\mansh\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\response.py:566: RuntimeWarning: coroutine 'async_generator_to_list' was never awaited
  with self._error_catcher():


Message 1: Tell me a joke on ML engineer
Response: Why did the machine learning engineer break up with their partner?

Because they had too many "overfitting" issues!
Time taken: 22.02 seconds

Total time for 1 messages: 22.02 seconds
Average time per message: 22.02 seconds


In [5]:
# for 5.65M model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process a list of stories
async def main():
    # Define your list of tuples (serial number, partial_story, completed_story)
    stories = [
        ("1", "In a dark basement, there is a white", "<start> In a dark basement, there is a white toilet and a shiny sink. The sink is so cool and smile and some are standing together. It feels like a fun place to play!<pad>"),
        ("2", "In a shiny bathroom, the walls sparkle like", "<start> In a shiny bathroom, the walls sparkle like a big bowl. The sign stands tall and some are standing together. It feels like a fun place to wash to go!<pad>"),
        ("3", "There is a big table full of yummy", "<start> There is a big table full of yummy food! They are so colorful fruits and some are smiling. They are all so happy and ready to eat!<pad>"),
        ("4", "Pink cakes and lollipops rest on white tables", "<start> Pink cakes and lollipops rest on white tables. They stand together in the sky, like a big book and smile. It looks like a fun place to play!<pad>"),
        ("5", "The cake is so colorful with chocolate and", "<start> The cake is so colorful with chocolate and some big bags. The sign stands tall and stand tall around them. It looks like a fun place to play!<pad>"),
        ("6", "In a funny bathroom, there are two shiny", "<start> In a funny bathroom, there are two shiny sinks. They stand together in the sky, like a big busy street. The sun shines bright and fun!<pad>"),
        ("7", "In a happy green bathroom, there are funny", "<start> In a happy green bathroom, there are funny streets. They sit on the sidewalk and some trees are standing together. It feels like a fun place to wash the carot is nearby!<pad>"),
        ("8", "The bathroom has a white toilet and a", "<start> The bathroom has a white toilet and a big sink. They are standing together, like a big bird flying on the sidewalk. It looks like a fun place to play!<pad>"),
        ("9", "In a shiny bathroom, there is a big", "<start> In a shiny bathroom, there is a big bridge. They are standing together, like a big blue busy street. The sun shines bright and smile!<pad>"),
        ("10", "There's a man on a shiny, old motorcycle", "<start> There's a man on a shiny, old motorcycle. They are standing together, like a big bird with their skis. The sun shines bright and smile!<pad>"),
        ("11", "There's a big building with a clock inside", "<start> There's a big building with a clock inside. They are standing together, like a big bird flying and some busy. The sun shines bright and fun!<pad>"),
        ("12", "The green bowl is on the table. It", "<start> The green bowl is on the table. It has a shiny sink and stands tall and stands tall around. The sun shines bright and smile!<pad>"),
        ("13", "There is a big, yummy cake on a", "<start> There is a big, yummy cake on a big boat. They are standing together, like a big bird with their soft green grass. It looks like a fun place to play!<pad>"),
        ("14", "A big parade is happening! A police motorcycle", "<start> A big parade is happening! A police motorcycle stands tall and smile. The sun shines bright and smile as they stand together.<pad>"),
        ("15", "A fluffy cat is on a table. It", "<start> A fluffy cat is on a table. It has a big brick and a big smile. The cat looks so cozy and fun!<pad>"),
        ("16", "The orange kitty sits on the table beside", "<start> The orange kitty sits on the table beside. It has a shiny sink and some bikes are waiting for the street. It looks like a fun place to play!<pad>"),
        ("17", "The kitty is very funny. It stands in", "<start> The kitty is very funny. It stands in a big tree station to sit next to a big store. The sign is so tasty and fun!<pad>"),
        ("18", "The cat is eating its food. It's funny", "<start> The cat is eating its food. It's funny street and some are soft bed with bright red bricks. The cat looks so tasty and fun!<pad>"),
        ("19", "A young man is sitting in a small", "<start> A young man is sitting in a small with a big smile. The cake is so cool and smile and some are smiling. It looks like a fun place to play!<pad>"),
        ("20", "The toilet has a big, round light above", "<start> The toilet has a big, round light above the street. They are all lined up and smiling and smile. It looks like a fun place to play!<pad>"),
        ("21", "In a tiny bathroom, there is a white", "<start> In a tiny bathroom, there is a white toilet and a shiny sink. The sink is soft and soft and soft and colorful too! It feels like a fun place to play!<pad>"),
        ("22", "There are tiny green beads and nuts inside", "<start> There are tiny green beads and nuts inside. They are standing together, like a big book and smile. The sun shines bright and smile!<pad>"),
        ("23", "A man sits at his desk with a", "<start> A man sits at his desk with a big bowl. The sign stands tall and the street sign that says 'TV.' The sun shines bright and smile!<pad>"),
        ("24", "The bowl has yummy fruit like apples, bananas,", "<start> The bowl has yummy fruit like apples, bananas, and the sink is so cool! They are all around them, ready to help them. It looks like a fun place to play!<pad>"),
        ("25", "In a big parking lot, two cool motorbikes", "<start> In a big parking lot, two cool motorbikes. The sun shines bright and the sidewalk around. The sun shines bright and smile as they stand together!<pad>")
    ]

    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    data = {
            "Partial": [story[1] for story in tasks],
            "Complete": [story[2] for story in tasks]
        }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("ShortDesc_5.65M_short_partial_results.csv")

    print("@" *100)
    print("\n printing the responses")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  Serial Number: 13
Grammar: 6/10
Creativity: 8/10
Consistency: 8/10
Meaningfulness: 7/10
Plot: 7/10

Total Score: 36/40

Age Group: A: 3 or under

The grammar could be improved, but the creativity and plot development are strong and show imaginative thinking typical for a 3-year-old child.
Generated response:  Serial Number: 3

Grammar: 8
Creativity: 9
Consistency: 10
Meaningfulness: 10
Plot: 9

Total Score: 46/50

Based on the language and creativity, the best guess for the age of the student is A: 3 or under.
Generated response:  Serial Number: 15

Grammar: 9/10
Creativity: 10/10
Consistency: 10/10
Meaningfulness: 8/10
Plot: 9/10
Total Score: 46/50

Age Group: B (4-5)
Generated response:  **Serial Number: 2**

**Partial Story:** 
In a shiny bathroom, the walls sparkle like

**Completed Story:** 
<start> In a shiny bathroom, the walls sparkle like a big bowl. The sign stands tall and some are standing together. It feels like a fun place to wash to go!<pad>

**Grade

In [6]:
# fr 16M model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process a list of stories
async def main():
    # Define your list of tuples (serial number, partial_story, completed_story)
    stories = [
    ("1", "In a dark basement, there is a white", "<start> In a dark basement, there is a white bathroom. The bathroom is bright and shiny. It looks like a fun place to play!<pad>"),
    ("2", "In a shiny bathroom, the walls sparkle like", "<start> In a shiny bathroom, the walls sparkle like stars. There’s a big white tub nearby, and a sink sparkles nearby. It feels like a fun place to splash and play!<pad>"),
    ("3", "There is a big table full of yummy,", "<start> There is a big table full of yummy food! There are shiny plates with colorful veggies, and tasty meat. It looks like a fun place to eat!<pad>"),
    ("4", "Pink cakes and lollipops rest on white tables,", "<start> Pink cakes and lollipops rest on white tables. The cake has a shiny candle on top, and the candles are so yummy! It’s like a fun party for the cake!<pad>"),
    ("5", "The cake is so colorful with chocolate and,", "<start> The cake is so colorful with chocolate and yummy fruit. It has a soft bread and a shiny toaster. I want to eat it all!<pad>"),
    ("6", "In a funny bathroom, there are two shiny", "<start> In a funny bathroom, there are two shiny sinks. They sit on a sink, all ready for a place to splash and play. It’s like a secret place to splash and play!<pad>"),
    ("7", "In a happy green bathroom, there are funny", "<start> In a happy green bathroom, there are funny wooden beans. They sit on a shiny toilet that looks like a secret place. It’s a silly place to splash and play!<pad>"),
    ("8", "The bathroom has a white toilet and a,", "<start> The bathroom has a white toilet and a shiny sink. The walls are bright yellow, and there are cool sinks and a special sink. It feels like a fun place to splash and play!<pad>"),
    ("9", "In a shiny bathroom, there is a big", "<start> In a shiny bathroom, there is a big tub and a sink. The walls are bright and green, and there are soft towels and a shower too! It feels like a secret place waiting to be used!<pad>"),
    ("10", "There's a man on a shiny, old motorcycle", "<start> There's a man on a shiny, old motorcycle. They stand together like friends waiting for a fun ride. The street is busy with cars and people.<pad>"),
    ("11", "There's a big building with a clock inside,", "<start> There's a big building with a clock inside. The clock is shining bright, and the clock is so black and white. The clock tells time for everyone the time.<pad>"),
    ("12", "The green bowl is on the table. It,", "<start> The green bowl is on the table. It has a big basket of yummy fruit and crunchy veggies. The broccoli looks so tasty and fun!<pad>"),
    ("13", "There is a big, yummy cake on a", "<start> There is a big, yummy cake on a plate. It has a soft bread and a shiny candle on top. It looks so sweet and fun!<pad>"),
    ("14", "A big parade is happening! A police motorcycle,", "<start> A big parade is happening! A police motorcycle is parked by a tall building. The street is full of life and a colorful bike nearby!<pad>"),
    ("15", "A fluffy cat is on a table. It,", "<start> A fluffy cat is on a table. It has a shiny laptop on it. The cat looks curious and a little screen!<pad>"),
    ("16", "The orange kitty sits on the table beside,", "<start> The orange kitty sits on the table beside the store. It has a shiny glass of wine and a cozy bed. The kitchen is nearby, ready for fun!<pad>"),
    ("17", "The kitty is very funny. It stands in,", "<start> The kitty is very funny. It stands in a big place with a shiny camera. The kitchen is ready for a special party!<pad>"),
    ("18", "The cat is eating its food. It's funny,", "<start> The cat is eating its food. It's funny and sits on the sidewalk, looking at the camera. The cat seems curious about the cat with a big smile!<pad>"),
    ("19", "A young man is sitting in a small,", "<start> A young man is sitting in a small boat. He looks at the big blue ocean and the water sparkles. It’s like a fun party!<pad>"),
    ("20", "The toilet has a big, round light above", "<start> The toilet has a big, round light above the side. It says 'E Mario St.' The sign says 'Stop Street St.' It makes me wonder what it sees!<pad>"),
    ("21", "In a tiny bathroom, there is a white", "<start> In a tiny bathroom, there is a white toilet and a shiny sink. The sink is a little mirror, and the counters are waiting. It feels like a fun place to splash and play!<pad>"),
    ("22", "There are tiny green beads and nuts inside,", "<start> There are tiny green beads and nuts inside. They sit on a big table with some books. The bears look so cute and cute!<pad>"),
    ("23", "A man sits at his desk with a,", "<start> A man sits at his desk with a shiny laptop. He looks at the screen where people walk by. It’s like a fun place to play!<pad>"),
    ("24", "The bowl has yummy fruit like apples, bananas,", "<start> The bowl has yummy fruit like apples, bananas, and bright oranges. There are also bright orange carrots and green broccoli too! It looks so tasty and fun to eat!<pad>"),
    ("25", "In a big parking lot, two cool motorbikes", "<start> In a big parking lot, two cool motorbikes stand together. They look like they are waiting for a fun ride. The street is busy with cars and a big bus are nearby!<pad>")
]



    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    data = {
            "Partial": [story[1] for story in tasks],
            "Complete": [story[2] for story in tasks]
        }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("ShortDesc_16M_short_partial_results.csv")

    print("@" *100)
    print("\n printing the responses")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  **Serial Number: 5**

**Completed Story:** "The cake is so colorful with chocolate and yummy fruit. It has a soft bread and a shiny toaster. I want to eat it all!"

**Grades:**
- Grammar: 7/10
- Creativity: 6/10
- Consistency: 5/10
- Meaningfulness: 6/10
- Plot: 5/10

**Total Score: 29/50**

**Estimated Age Group:** B: 4-5

**Comments:**
- **Grammar**: The sentence is mostly grammatically correct, but "soft bread" and "shiny toaster" within the context of a cake is a bit awkward.
- **Creativity**: Including "yummy fruit" shows a bit of creativity, but "soft bread" and "shiny toaster" seem out of place.
- **Consistency**: The addition of "shiny toaster" breaks consistency as it does not relate directly to a cake.
- **Meaningfulness**: The sentence makes sense overall, but the inclusion of "toaster" affects its meaningfulness.
- **Plot**: The plot does not develop much; it merely describes the cake and expresses a desire to eat it.
Generated response:  Serial Number:

In [7]:
## for 25M model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process a list of stories
async def main():
    # Define your list of tuples (serial number, partial_story, completed_story)
    stories = [
    ("1", "In a dark basement, there is a white", "In a dark basement, there is a white frisbee with a big bag. The basket is full of yummy food and drinks are waiting. It feels like a fun party!<pad>"),
    ("2", "In a shiny bathroom, the walls sparkle like", "In a shiny bathroom, the walls sparkle like the sky. There’s a big mirror that makes me smile. The walls are soft and white, making it feel cozy!<pad>"),
    ("3", "There is a big table full of yummy,", "There is a big table full of yummy food! There are colorful fruits and vegetables on top. It looks like a fun party for my tummy!<pad>"),
    ("4", "Pink cakes and lollipops rest on white tables,", "Pink cakes and lollipops rest on white tables. They are all different shapes and sizes. It’s like a tiny adventure is waiting to be eaten!<pad>"),
    ("5", "The cake is so colorful with chocolate and,", "The cake is so colorful with chocolate and shiny candles. There are so many cans and a big candle on top. It looks like a fun party!<pad>"),
    ("6", "In a funny bathroom, there are two shiny", "In a funny bathroom, there are two shiny toilets! They are ready to help kids need to eat yummy food. It’s like a special place for books!<pad>"),
    ("7", "In a happy green bathroom, there are funny", "In a happy green bathroom, there are funny toilets! They are all different colors and shapes. It looks like a secret place for girls!<pad>"),
    ("8", "The bathroom has a white toilet and a,", "The bathroom has a white toilet and a shiny sink. The walls are bright and full of colorful tiles. There’s also a sink and a toilet too!<pad>"),
    ("9", "In a shiny bathroom, there is a big", "In a shiny bathroom, there is a big tub for splashes and a little mane. The walls are soft and white, making it feel cozy. It’s like a fun place to splash and play!<pad>"),
    ("10", "There's a man on a shiny, old motorcycle", "There's a man on a shiny, old motorcycle. He wears a cool helmet and smiles big. He looks ready to zoom away!<pad>"),
    ("11", "There's a big building with a clock inside,", "There's a big building with a clock inside. The clock tells time for everyone. It stands tall in the city, watching over the streets.<pad>"),
    ("12", "The green bowl is on the table. It,", "The green bowl is on the table. It has yummy beans and bright colors like red anbanas. The broccoli looks so tasty and fun!<pad>"),
    ("13", "There is a big, yummy cake on a", "There is a big, yummy cake on a table! There are shiny chocolate cakes and a shiny fork all over it. It looks like a fun party is about to eat!<pad>"),
    ("14", "A big parade is happening! A police motorcycle,", "A big parade is happening! A police motorcycle shines in the airplane, and a man is waiting for its turn. The sky is bright and blue!<pad>"),
    ("15", "A fluffy cat is on a table. It,", "A fluffy cat is on a table. It looks so cozy next to a big TV! The cat is watching the world outside.<pad>"),
    ("16", "The orange kitty sits on the table beside,", "The orange kitty sits on the table beside it. There are shiny candles and a big candle on top. The cabinets look funny and cozy!<pad>"),
    ("17", "The kitty is very funny. It stands in,", "The kitty is very funny. It stands in the woods, like a big fridge and stories. The cabinet is cozy and sleepy, waiting for adventures!<pad>"),
    ("18", "The cat is eating its food. It's funny,", "The cat is eating its food. It's funny and sitting on the counter. The cat looks so happy and curious!<pad>"),
    ("19", "A young man is sitting in a small,", "A young man is sitting in a small chair. He has a big box of shiny green bags and a picture. He looks happy and ready for a fun time!<pad>"),
    ("20", "The toilet has a big, round light above", "The toilet has a big, round light above it. There are two boxes of shiny buttons all around. A soft tower is nearby, too!<pad>"),
    ("21", "In a tiny bathroom, there is a white", "In a tiny bathroom, there is a white toilet and a big tub. The toilet is also sitting on a shiny toilet. It feels like a fun place to splash!<pad>"),
    ("22", "There are tiny green beads and nuts inside,", "There are tiny green beads and nuts inside. They are all together on a big black chair. The bed is cozy and warm!<pad>"),
    ("23", "A man sits at his desk with a,", "A man sits at his desk with a big computer and a little kid. The desk is full of fun things to show the computer donuts!<pad>"),
    ("24", "The bowl has yummy fruit like apples, bananas,", "The bowl has yummy fruit like apples, bananas, and bright oranges. There are also a big brown bananas on the table. It looks like a fun meal!<pad>"),
    ("25", "In a big parking lot, two cool motorbikes", "In a big parking lot, two cool motorbikes stand tall on the sidewalk. They look like friends waiting for a friend to go on an adventure! The big building stands tall at the bright sky.<pad>")
]



    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    data = {
            "Partial": [story[1] for story in tasks],
            "Complete": [story[2] for story in tasks]
        }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("ShortDesc_25M_short_partial_results.csv")

    print("@" *100)
    print("\n printing the responses")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  **Serial Number: 4**

**Partial Story:**
Pink cakes and lollipops rest on white tables,

**Completed Story:**
Pink cakes and lollipops rest on white tables. They are all different shapes and sizes. It’s like a tiny adventure is waiting to be eaten!

**Grading:**

- **Grammar (10/10):** The text is grammatically correct, with appropriate punctuation and sentence structure.
- **Creativity (9/10):** The idea of treats being a "tiny adventure" is both imaginative and engaging.
- **Consistency (10/10):** The continuation is consistent with the beginning and flows smoothly.
- **Meaningfulness (8/10):** The text is meaningful and paints a vivid picture, though it could benefit from a bit more detail.
- **Plot (7/10):** While there isn't a complex plot, the completion does add a sense of excitement and adventure, suitable for the age group.

**Total Score: 44/50**

**Estimated Age Group: B: 4-5**

My best guess is that the student is within the 4-5 age range. The creativit